# Food Image Classification with EfficientNetB3
### Transfer Learning on the `bjoernjostein/food-classification` Dataset (61 Classes)

## Prerequisites

Before running this notebook, ensure you have Kaggle API credentials configured:
1. Go to https://www.kaggle.com/settings/api
2. Click **Create New Token** — this downloads `kaggle.json`
3. Place `kaggle.json` at `~/.kaggle/kaggle.json` (Linux/Mac) or `C:\\Users\\<you>\\.kaggle\\kaggle.json` (Windows)
4. On Linux/Mac: `chmod 600 ~/.kaggle/kaggle.json`

The notebook uses `kagglehub` which reads these credentials automatically.

In [ ]:
!pip install kagglehub timm torch torchvision numpy pandas matplotlib seaborn scikit-learn Pillow tqdm --quiet

## Section 1 — Setup & Imports

In [ ]:
import os
import random
import warnings
import pathlib

import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, cohen_kappa_score
)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
import torchvision.transforms as T
import timm

warnings.filterwarnings('ignore')

In [ ]:
SEED         = 42
DATASET_SLUG = "bjoernjostein/food-classification"
IMG_SIZE     = (224, 224)
BATCH_SIZE   = 32
EPOCHS       = 5 if not torch.cuda.is_available() else 20
LR           = 1e-4
FINE_TUNE_LR = 1e-5
NUM_WORKERS  = 0   # 0 is safest on Windows; increase if on Linux
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Config: EPOCHS={EPOCHS}, BATCH_SIZE={BATCH_SIZE}, DEVICE={DEVICE}")
if not torch.cuda.is_available():
    print("⚠️  No GPU detected. Training will be slow. Consider using Google Colab or reducing EPOCHS.")
else:
    print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")

In [ ]:
def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
os.makedirs("figures", exist_ok=True)
print("Seed set. figures/ directory ready.")